In [1]:
# Install packages
!pip install dspy-ai anthropic --quiet

In [2]:
import os
import sys

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Add project to path
UTILS_DIR = '/content/drive/MyDrive/AI6129/codes/utilities'

if UTILS_DIR not in sys.path:
    sys.path.append(UTILS_DIR)

# Verify the file exists
py_file = os.path.join(UTILS_DIR, "isolate_granularity_diagnostic.py")
print(f"Python file exists: {os.path.exists(py_file)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python file exists: True


In [3]:
# UPDATE THESE PATHS TO MATCH YOUR DIRECTORY STRUCTURE
PROJECT_DIR = "/content/drive/MyDrive/AI6129"

# Directory containing individual PMCxxxxxxx.json ground truth files
GT_DIR = os.path.join(PROJECT_DIR, "ground_truth")  # UPDATE THIS

# Path to the stratified splits metadata file
SPLITS_PATH = os.path.join(PROJECT_DIR, "assay/validation_splits/assay_tadp_gepa_optimised_splits.json")  # UPDATE THIS

# Directory containing article XML files
XML_DIR = os.path.join(PROJECT_DIR, "xml")  # UPDATE THIS

# Output directory for diagnostic CSVs
OUTPUT_DIR = os.path.join(PROJECT_DIR, "assay/diagnostic_output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths
print("Path Verification:")
print(f"  GT Directory: {GT_DIR}")
print(f"    Exists: {os.path.exists(GT_DIR)}")
if os.path.exists(GT_DIR):
    json_count = len([f for f in os.listdir(GT_DIR) if f.startswith("PMC") and f.endswith(".json")])
    print(f"    PMC JSON files found: {json_count}")

print(f"  Splits File: {SPLITS_PATH}")
print(f"    Exists: {os.path.exists(SPLITS_PATH)}")

print(f"  XML Directory: {XML_DIR}")
print(f"    Exists: {os.path.exists(XML_DIR)}")

print(f"  Output Directory: {OUTPUT_DIR}")
print(f"    Exists: {os.path.exists(OUTPUT_DIR)}")

Path Verification:
  GT Directory: /content/drive/MyDrive/AI6129/ground_truth
    Exists: True
    PMC JSON files found: 227
  Splits File: /content/drive/MyDrive/AI6129/assay/validation_splits/assay_tadp_gepa_optimised_splits.json
    Exists: True
  XML Directory: /content/drive/MyDrive/AI6129/xml
    Exists: True
  Output Directory: /content/drive/MyDrive/AI6129/assay/diagnostic_output
    Exists: True


In [4]:
import dspy
from google.colab import userdata

# Option 1: Anthropic Claude
try:
    api_key = userdata.get('CLAUDE_API_KEY')
    lm = dspy.LM(
        model="anthropic/claude-haiku-4-5-20251001",
        api_key=api_key,
        max_tokens=16384
    )
    print("Using Anthropic Claude")
except:
    # Option 2: Google Gemini
    lm = dspy.LM(model="gemini/gemini-1.5-flash")
    print("Using Google Gemini")

dspy.configure(lm=lm)

Using Anthropic Claude


In [5]:
import json
# Import the diagnostic module
from isolate_granularity_diagnostic import (
    run_diagnostic_pipeline,
    print_pattern_summary,
    load_ground_truth
)

# Test run with 5 documents
results, ground_truth = run_diagnostic_pipeline(
    ground_truth_path=GT_DIR,
    xml_directory=XML_DIR,
    output_directory=OUTPUT_DIR,
    splits_path=SPLITS_PATH,
    max_documents=5  # Start small
)

print_pattern_summary(results, ground_truth)

[INFO] Loading ground truth...
[INFO] Loading GT from directory: /content/drive/MyDrive/AI6129/ground_truth
[INFO] Loaded splits metadata for 227 documents
[INFO] Found 227 PMC JSON files
[INFO] Loaded ground truth for 227 documents
[INFO] Loaded 227 documents from ground truth
[INFO] Processing 5 documents
[INFO] Processing 1/5: PMC4672624
[INFO] Extracted 2 isolates from PMC4672624
[INFO] Processing 2/5: PMC4806280
[INFO] Extracted 1 isolates from PMC4806280
[INFO] Processing 3/5: PMC4824889
[INFO] Extracted 0 isolates from PMC4824889
[INFO] Processing 4/5: PMC4859176
[INFO] Extracted 26 isolates from PMC4859176
[INFO] Processing 5/5: PMC4866840
[INFO] Extracted 7 isolates from PMC4866840
[INFO] Wrote 36 rows to /content/drive/MyDrive/AI6129/assay/diagnostic_output/discrepancy_analysis_20260201_021721.csv
[INFO] Wrote 5 document summaries to /content/drive/MyDrive/AI6129/assay/diagnostic_output/document_summary_20260201_021721.csv
[INFO] Wrote 5 granularity analyses to /content/drive

In [6]:
# Run on validation set only (for focused analysis)
import json

# Load splits to get validation PMCIDs
with open(SPLITS_PATH, 'r') as f:
    splits = json.load(f)

validation_pmcids = splits.get("validation_set", [])
print(f"Validation set contains {len(validation_pmcids)} documents")

# Run on validation set
results, ground_truth = run_diagnostic_pipeline(
    ground_truth_path=GT_DIR,
    xml_directory=XML_DIR,
    output_directory=OUTPUT_DIR,
    splits_path=SPLITS_PATH,
    pmcid_list=validation_pmcids  # Only process validation set
)

print_pattern_summary(results, ground_truth)

Validation set contains 35 documents
[INFO] Loading ground truth...
[INFO] Loading GT from directory: /content/drive/MyDrive/AI6129/ground_truth
[INFO] Loaded splits metadata for 227 documents
[INFO] Found 227 PMC JSON files
[INFO] Loaded ground truth for 227 documents
[INFO] Loaded 227 documents from ground truth
[INFO] Processing 35 documents
[INFO] Processing 1/35: PMC6035434
[INFO] Extracted 6 isolates from PMC6035434
[INFO] Processing 2/35: PMC9216381
[INFO] Extracted 12 isolates from PMC9216381
[INFO] Processing 3/35: PMC9513250
[INFO] Extracted 11 isolates from PMC9513250
[INFO] Processing 4/35: PMC8311177
[INFO] Extracted 6 isolates from PMC8311177
[INFO] Processing 5/35: PMC9708683
[INFO] Extracted 16 isolates from PMC9708683
[INFO] Processing 6/35: PMC8784875
[INFO] Extracted 11 isolates from PMC8784875
[INFO] Processing 7/35: PMC9739812
[INFO] Extracted 18 isolates from PMC9739812
[INFO] Processing 8/35: PMC9309525
[INFO] Extracted 3 isolates from PMC9309525
[INFO] Processin